## SARIMA EN GASTOS OPERATIVOS


In [8]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_absolute_error, mean_squared_error
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

# ── 1. Datos 
df = pd.read_csv('Gastos.csv', encoding='latin1')
df.rename(columns={df.columns[0]: 'Fecha'}, inplace=True)
df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
# Columnas que componen Gastos Operativos
col_refac  = [c for c in df.columns if 'Costo de Venta (Refacciones)'    in c][0]
col_llanta = [c for c in df.columns if 'Costo de llantas'   in c][0]
col_lubric = [c for c in df.columns if 'Costo de Lubricantes'   in c][0]
col_op     = [c for c in df.columns if 'Gastos de OperaciÃ³n'   in c][0]

print("Columnas seleccionadas:")
for c in [col_refac, col_llanta, col_lubric, col_op]:
    print(f"  · {c}")

df_idx = df.set_index('Fecha')
df_idx['Gastos Operativos'] = (df_idx[col_refac] +
                                df_idx[col_llanta] +
                                df_idx[col_lubric] +
                                df_idx[col_op])
ts = df_idx['Gastos Operativos'].asfreq('MS')
ts['2021-12-01'] = (ts['2021-11-01'] + ts['2022-01-01']) / 2  # corrige negativo dic-21

train = ts['2021':'2024']
test  = ts['2025']
print(f"Train: {train.index[0].date()} → {train.index[-1].date()} ({len(train)} meses)")
print(f"Test : {test.index[0].date()} → {test.index[-1].date()} ({len(test)} meses)")

# ── 2. Mejor modelo: SARIMA(1,1,1)(1,0,1)[12] 
print("\nAjustando SARIMA(1,1,1)(1,0,1)[12] …")
model  = SARIMAX(train, order=(1,1,1), seasonal_order=(1,0,1,12),
                 enforce_stationarity=False, enforce_invertibility=False)
result = model.fit(disp=False, maxiter=300)
print(result.summary())

fc_obj  = result.get_forecast(steps=12)
fc_mean = fc_obj.predicted_mean
fc_ci   = fc_obj.conf_int(alpha=0.05)
fc_mean.index = test.index
fc_ci.index   = test.index

mae  = mean_absolute_error(test, fc_mean)
rmse = np.sqrt(mean_squared_error(test, fc_mean))
mape = np.mean(np.abs((test - fc_mean) / test)) * 100
print(f"\nMAE : ${mae:,.0f}  |  RMSE: ${rmse:,.0f}  |  MAPE: {mape:.1f}%")

fitted = result.fittedvalues

# Componente estacional de la descomposición
decomp  = seasonal_decompose(train, model='additive', period=12)
MESES   = ['Ene','Feb','Mar','Abr','May','Jun',
           'Jul','Ago','Sep','Oct','Nov','Dic']
sea_avg = pd.Series(decomp.seasonal.values, index=decomp.seasonal.index)
sea_mes = sea_avg.groupby(sea_avg.index.month).mean()
sea_mes.index = MESES

# Pronóstico 2026 (reentrenar con todo)
model2  = SARIMAX(ts, order=(1,1,1), seasonal_order=(1,0,1,12),
                  enforce_stationarity=False, enforce_invertibility=False)
res2    = model2.fit(disp=False, maxiter=300)
fc2_obj = res2.get_forecast(steps=12)
fc2     = fc2_obj.predicted_mean
fc2_ci  = fc2_obj.conf_int(alpha=0.05)
fc2.index    = pd.date_range('2026-01-01', periods=12, freq='MS')
fc2_ci.index = fc2.index

# ── 3. Paleta ─────────────────────────────────────────────────────────────────
BLUE  = '#1C4E80'
ORAN  = '#E8872A'
GREEN = '#21A659'
RED   = '#E74C3C'
GRAY  = '#B0B8C1'
BG    = '#F4F7FB'
CARD  = '#FFFFFF'
fmt_M = mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M')

fig = plt.figure(figsize=(20, 22), facecolor=BG)
fig.suptitle('SARIMA(1,1,1)(1,0,1)[12] — Gastos Operativos\n(Refacciones + Llantas + Lubricantes + Operación)',
             fontsize=19, fontweight='bold', color=BLUE, y=0.987)
gs = fig.add_gridspec(4, 2, hspace=0.52, wspace=0.30,
                      left=0.08, right=0.96, top=0.96, bottom=0.04)

# ── A: Serie + ajuste in-sample ───────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor(CARD)
ax1.plot(train.index, train.values/1e6, color=GRAY, lw=1.5,
         alpha=0.9, label='Real (train 2021–2024)')
ax1.plot(fitted.index, fitted.values/1e6, color=BLUE, lw=1.3,
         alpha=0.75, linestyle='-', label='Ajuste in-sample')
ax1.set_title('Serie histórica Gastos Operativos con ajuste del modelo',
              fontsize=12, fontweight='bold', color=BLUE, pad=8)
ax1.set_ylabel('Millones MXN', fontsize=10)
ax1.yaxis.set_major_formatter(fmt_M)
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax1.tick_params(axis='x', labelsize=8)
ax1.legend(fontsize=9, loc='upper left', framealpha=0.7)
ax1.grid(axis='y', linestyle='--', alpha=0.35)
ax1.spines[['top','right']].set_visible(False)
for yr in ['2022-01-01','2023-01-01','2024-01-01']:
    ax1.axvline(pd.Timestamp(yr), color=BLUE, lw=0.6, linestyle=':', alpha=0.3)

# ── B: Pronóstico 2025 vs Real ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, :])
ax2.set_facecolor(CARD)
ctx = train[-6:]
ax2.plot(ctx.index, ctx.values/1e6, color=GRAY, lw=1.3,
         alpha=0.5, label='Real (jul–dic 2024)')
ax2.fill_between(fc_ci.index,
                 fc_ci.iloc[:,0]/1e6, fc_ci.iloc[:,1]/1e6,
                 color=ORAN, alpha=0.18, label='IC 95%')
ax2.plot(fc_mean.index, fc_mean.values/1e6,
         color=ORAN, lw=2.3, linestyle='--', label='Pronóstico SARIMA')
ax2.plot(test.index, test.values/1e6,
         color=BLUE, lw=2.0, marker='o', markersize=6, label='Real 2025')
ax2.set_title('Pronóstico SARIMA vs Real 2025 (validación)',
              fontsize=12, fontweight='bold', color=BLUE, pad=8)
ax2.set_ylabel('Millones MXN', fontsize=10)
ax2.yaxis.set_major_formatter(fmt_M)
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax2.tick_params(axis='x', labelsize=8)
ax2.legend(fontsize=9, framealpha=0.8, loc='upper right')
ax2.grid(axis='y', linestyle='--', alpha=0.35)
ax2.spines[['top','right']].set_visible(False)
txt = f"MAE = ${mae/1e6:.2f}M   RMSE = ${rmse/1e6:.2f}M   MAPE = {mape:.1f}%"
ax2.text(0.01, 0.97, txt, transform=ax2.transAxes, ha='left',
         fontsize=9.5, va='top',
         bbox=dict(boxstyle='round,pad=0.35', fc='#FFF8EC', ec=ORAN, lw=1.2))

# ── C: Componente estacional por mes ─────────────────────────────────────────
ax3 = fig.add_subplot(gs[2, 0])
ax3.set_facecolor(CARD)
bar_colors = [RED   if v == sea_mes.min() else
              GREEN if v == sea_mes.max() else
              BLUE  for v in sea_mes.values]
bars = ax3.bar(sea_mes.index, sea_mes.values/1e6,
               color=bar_colors, edgecolor='white',
               linewidth=0.6, width=0.65, zorder=3)
ax3.axhline(0, color=GRAY, lw=1, linestyle='--')
ax3.set_title('Componente estacional promedio\npor mes del año',
              fontsize=11, fontweight='bold', color=BLUE, pad=8)
ax3.set_ylabel('Desviación vs tendencia (M MXN)', fontsize=10)
ax3.yaxis.set_major_formatter(fmt_M)
ax3.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)
ax3.spines[['top','right']].set_visible(False)
for bar, val in zip(bars, sea_mes.values):
    offset = 0.08 if val >= 0 else -0.22
    ax3.text(bar.get_x()+bar.get_width()/2, val/1e6+offset,
             f'${val/1e6:.1f}M', ha='center', va='bottom',
             fontsize=7.5, fontweight='bold', color='#222')
mejor_mes = sea_mes.idxmax()
ax3.text(0.5, -0.18, f'📈  Mes con mayor gasto estacional: {mejor_mes}',
         transform=ax3.transAxes, ha='center',
         fontsize=10, fontweight='bold', color=RED)

# ── D: Residuales pronóstico 2025 ─────────────────────────────────────────────
resid = test - fc_mean
ax4   = fig.add_subplot(gs[2, 1])
ax4.set_facecolor(CARD)
rc = [GREEN if r >= 0 else RED for r in resid.values]
ax4.bar(range(len(resid)), resid.values/1e6, color=rc,
        edgecolor='white', linewidth=0.4, width=0.65, zorder=3)
ax4.axhline(0, color=GRAY, lw=1, linestyle='--')
ax4.set_xticks(range(len(resid)))
ax4.set_xticklabels(MESES, fontsize=9)
ax4.set_title('Error del pronóstico\n(Real − Pronóstico) 2025',
              fontsize=11, fontweight='bold', color=BLUE, pad=8)
ax4.set_ylabel('Error (M MXN)', fontsize=10)
ax4.yaxis.set_major_formatter(fmt_M)
ax4.grid(axis='y', linestyle='--', alpha=0.35, zorder=0)
ax4.spines[['top','right']].set_visible(False)
ax4.legend(handles=[Patch(color=GREEN, label='Real > Pronóstico'),
                    Patch(color=RED,   label='Real < Pronóstico')],
           fontsize=8, framealpha=0.7)

# ── E: Pronóstico 2026 ────────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[3, :])
ax5.set_facecolor(CARD)
ctx2 = ts[-12:]
ax5.plot(ctx2.index, ctx2.values/1e6, color=GRAY, lw=1.5,
         alpha=0.7, label='Real 2025')
ax5.fill_between(fc2_ci.index,
                 fc2_ci.iloc[:,0]/1e6, fc2_ci.iloc[:,1]/1e6,
                 color=GREEN, alpha=0.15, label='IC 95%')
ax5.plot(fc2.index, fc2.values/1e6,
         color=GREEN, lw=2.3, linestyle='--',
         marker='o', markersize=5, label='Pronóstico 2026')
ax5.axvline(pd.Timestamp('2026-01-01'), color=GRAY,
            lw=1, linestyle=':', alpha=0.6)
ax5.set_title('Pronóstico Gastos Operativos 2026 (modelo 2021–2025)',
              fontsize=12, fontweight='bold', color=BLUE, pad=8)
ax5.set_ylabel('Millones MXN', fontsize=10)
ax5.yaxis.set_major_formatter(fmt_M)
ax5.xaxis.set_major_locator(mdates.MonthLocator())
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax5.tick_params(axis='x', labelsize=8)
ax5.legend(fontsize=9, framealpha=0.8)
ax5.grid(axis='y', linestyle='--', alpha=0.35)
ax5.spines[['top','right']].set_visible(False)
for date, val in fc2.items():
    ax5.text(date, val/1e6+0.2, f'${val/1e6:.1f}M',
             ha='center', va='bottom', fontsize=7.5,
             color=GREEN, fontweight='bold')

plt.savefig('gastos_operacion.png',
            dpi=150, bbox_inches='tight', facecolor=BG)
print("\n✅  Guardado.")

print("\n── Pronóstico Gastos Operativos 2026 ──")
for d, v in fc2.items():
    print(f"  {d.strftime('%b %Y')}: ${v:>12,.0f}")
print(f"\n  Total anual estimado 2026: ${fc2.sum():,.0f}")

Columnas seleccionadas:
  · Costo de Venta (Refacciones)
  · Costo de llantas
  · Costo de Lubricantes
  · Gastos de OperaciÃ³n
Train: 2021-07-01 → 2024-12-01 (42 meses)
Test : 2025-01-01 → 2025-12-01 (12 meses)

Ajustando SARIMA(1,1,1)(1,0,1)[12] …
                                     SARIMAX Results                                      
Dep. Variable:                  Gastos Operativos   No. Observations:                   42
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 12)   Log Likelihood                -439.342
Date:                            Sat, 21 Mar 2026   AIC                            888.684
Time:                                    00:21:08   BIC                            895.163
Sample:                                07-01-2021   HQIC                           890.610
                                     - 12-01-2024                                         
Covariance Type:                              opg                                         
                 coef 

In [11]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_absolute_error, mean_squared_error
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

# ── 1. Datos 
df = pd.read_csv('Gastos.csv', encoding='latin1')
df.rename(columns={df.columns[0]: 'Fecha'}, inplace=True)
df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
# Columnas que componen Gastos Administrativos
col_mant  = [c for c in df.columns if 'Gastos de Mantenimiento'    in c][0]
col_admin = [c for c in df.columns if 'Gastos Administrativos'   in c][0]
col_fin= [c for c in df.columns if 'Gastos Financieros'   in c][0]
col_nf     = [c for c in df.columns if 'Gastos no fiscales'   in c][0]
col_ISR     = [c for c in df.columns if 'ISR  Ejercicio'   in c][0]
col_ISR_FA    = [c for c in df.columns if 'ISR Facilidades Administrativas'   in c][0]

print("Columnas seleccionadas:")
for c in [col_mant, col_admin, col_fin, col_nf,col_ISR,col_ISR_FA]:
    print(f"  · {c}")

df_idx = df.set_index('Fecha')
df_idx['Gastos Administrativos'] = (df_idx[col_mant] +
                                df_idx[col_admin] +
                                df_idx[col_fin] +
                                df_idx[col_nf]+
                                df_idx[col_ISR]+
                                df_idx[col_ISR_FA])

ts = df_idx['Gastos Administrativos'].asfreq('MS')
ts['2021-12-01'] = (ts['2021-11-01'] + ts['2022-01-01']) / 2  # corrige negativo dic-21

train = ts['2021':'2024']
test  = ts['2025']
print(f"Train: {train.index[0].date()} → {train.index[-1].date()} ({len(train)} meses)")
print(f"Test : {test.index[0].date()} → {test.index[-1].date()} ({len(test)} meses)")

# ── 2. Mejor modelo: SARIMA(1,1,1)(1,0,1)[12] 
print("\nAjustando SARIMA(1,1,1)(1,0,1)[12] …")
model  = SARIMAX(train, order=(1,1,1), seasonal_order=(1,0,1,12),
                 enforce_stationarity=False, enforce_invertibility=False)
result = model.fit(disp=False, maxiter=300)
print(result.summary())

fc_obj  = result.get_forecast(steps=12)
fc_mean = fc_obj.predicted_mean
fc_ci   = fc_obj.conf_int(alpha=0.05)
fc_mean.index = test.index
fc_ci.index   = test.index

mae  = mean_absolute_error(test, fc_mean)
rmse = np.sqrt(mean_squared_error(test, fc_mean))
mape = np.mean(np.abs((test - fc_mean) / test)) * 100
print(f"\nMAE : ${mae:,.0f}  |  RMSE: ${rmse:,.0f}  |  MAPE: {mape:.1f}%")

fitted = result.fittedvalues

# Componente estacional de la descomposición
decomp  = seasonal_decompose(train, model='additive', period=12)
MESES   = ['Ene','Feb','Mar','Abr','May','Jun',
           'Jul','Ago','Sep','Oct','Nov','Dic']
sea_avg = pd.Series(decomp.seasonal.values, index=decomp.seasonal.index)
sea_mes = sea_avg.groupby(sea_avg.index.month).mean()
sea_mes.index = MESES

# Pronóstico 2026 (reentrenar con todo)
model2  = SARIMAX(ts, order=(1,1,1), seasonal_order=(1,0,1,12),
                  enforce_stationarity=False, enforce_invertibility=False)
res2    = model2.fit(disp=False, maxiter=300)
fc2_obj = res2.get_forecast(steps=12)
fc2     = fc2_obj.predicted_mean
fc2_ci  = fc2_obj.conf_int(alpha=0.05)
fc2.index    = pd.date_range('2026-01-01', periods=12, freq='MS')
fc2_ci.index = fc2.index

# ── 3. Paleta ─────────────────────────────────────────────────────────────────
BLUE  = '#1C4E80'
ORAN  = '#E8872A'
GREEN = '#21A659'
RED   = '#E74C3C'
GRAY  = '#B0B8C1'
BG    = '#F4F7FB'
CARD  = '#FFFFFF'
fmt_M = mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M')

fig = plt.figure(figsize=(20, 22), facecolor=BG)
fig.suptitle('SARIMA(1,1,1)(1,0,1)[12] — Gastos Administrativos)',
             fontsize=19, fontweight='bold', color=BLUE, y=0.987)
gs = fig.add_gridspec(4, 2, hspace=0.52, wspace=0.30,
                      left=0.08, right=0.96, top=0.96, bottom=0.04)

# ── A: Serie + ajuste in-sample ───────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor(CARD)
ax1.plot(train.index, train.values/1e6, color=GRAY, lw=1.5,
         alpha=0.9, label='Real (train 2021–2024)')
ax1.plot(fitted.index, fitted.values/1e6, color=BLUE, lw=1.3,
         alpha=0.75, linestyle='-', label='Ajuste in-sample')
ax1.set_title('Serie histórica Gastos Administrativos con ajuste del modelo',
              fontsize=12, fontweight='bold', color=BLUE, pad=8)
ax1.set_ylabel('Millones MXN', fontsize=10)
ax1.yaxis.set_major_formatter(fmt_M)
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax1.tick_params(axis='x', labelsize=8)
ax1.legend(fontsize=9, loc='upper left', framealpha=0.7)
ax1.grid(axis='y', linestyle='--', alpha=0.35)
ax1.spines[['top','right']].set_visible(False)
for yr in ['2022-01-01','2023-01-01','2024-01-01']:
    ax1.axvline(pd.Timestamp(yr), color=BLUE, lw=0.6, linestyle=':', alpha=0.3)

# ── B: Pronóstico 2025 vs Real ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, :])
ax2.set_facecolor(CARD)
ctx = train[-6:]
ax2.plot(ctx.index, ctx.values/1e6, color=GRAY, lw=1.3,
         alpha=0.5, label='Real (jul–dic 2024)')
ax2.fill_between(fc_ci.index,
                 fc_ci.iloc[:,0]/1e6, fc_ci.iloc[:,1]/1e6,
                 color=ORAN, alpha=0.18, label='IC 95%')
ax2.plot(fc_mean.index, fc_mean.values/1e6,
         color=ORAN, lw=2.3, linestyle='--', label='Pronóstico SARIMA')
ax2.plot(test.index, test.values/1e6,
         color=BLUE, lw=2.0, marker='o', markersize=6, label='Real 2025')
ax2.set_title('Pronóstico SARIMA vs Real 2025 (validación)',
              fontsize=12, fontweight='bold', color=BLUE, pad=8)
ax2.set_ylabel('Millones MXN', fontsize=10)
ax2.yaxis.set_major_formatter(fmt_M)
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax2.tick_params(axis='x', labelsize=8)
ax2.legend(fontsize=9, framealpha=0.8, loc='upper right')
ax2.grid(axis='y', linestyle='--', alpha=0.35)
ax2.spines[['top','right']].set_visible(False)
txt = f"MAE = ${mae/1e6:.2f}M   RMSE = ${rmse/1e6:.2f}M   MAPE = {mape:.1f}%"
ax2.text(0.01, 0.97, txt, transform=ax2.transAxes, ha='left',
         fontsize=9.5, va='top',
         bbox=dict(boxstyle='round,pad=0.35', fc='#FFF8EC', ec=ORAN, lw=1.2))

# ── C: Componente estacional por mes ─────────────────────────────────────────
ax3 = fig.add_subplot(gs[2, 0])
ax3.set_facecolor(CARD)
bar_colors = [RED   if v == sea_mes.min() else
              GREEN if v == sea_mes.max() else
              BLUE  for v in sea_mes.values]
bars = ax3.bar(sea_mes.index, sea_mes.values/1e6,
               color=bar_colors, edgecolor='white',
               linewidth=0.6, width=0.65, zorder=3)
ax3.axhline(0, color=GRAY, lw=1, linestyle='--')
ax3.set_title('Componente estacional promedio\npor mes del año',
              fontsize=11, fontweight='bold', color=BLUE, pad=8)
ax3.set_ylabel('Desviación vs tendencia (M MXN)', fontsize=10)
ax3.yaxis.set_major_formatter(fmt_M)
ax3.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)
ax3.spines[['top','right']].set_visible(False)
for bar, val in zip(bars, sea_mes.values):
    offset = 0.08 if val >= 0 else -0.22
    ax3.text(bar.get_x()+bar.get_width()/2, val/1e6+offset,
             f'${val/1e6:.1f}M', ha='center', va='bottom',
             fontsize=7.5, fontweight='bold', color='#222')
mejor_mes = sea_mes.idxmax()
ax3.text(0.5, -0.18, f'📈  Mes con mayor gasto estacional: {mejor_mes}',
         transform=ax3.transAxes, ha='center',
         fontsize=10, fontweight='bold', color=RED)

# ── D: Residuales pronóstico 2025 ─────────────────────────────────────────────
resid = test - fc_mean
ax4   = fig.add_subplot(gs[2, 1])
ax4.set_facecolor(CARD)
rc = [GREEN if r >= 0 else RED for r in resid.values]
ax4.bar(range(len(resid)), resid.values/1e6, color=rc,
        edgecolor='white', linewidth=0.4, width=0.65, zorder=3)
ax4.axhline(0, color=GRAY, lw=1, linestyle='--')
ax4.set_xticks(range(len(resid)))
ax4.set_xticklabels(MESES, fontsize=9)
ax4.set_title('Error del pronóstico\n(Real − Pronóstico) 2025',
              fontsize=11, fontweight='bold', color=BLUE, pad=8)
ax4.set_ylabel('Error (M MXN)', fontsize=10)
ax4.yaxis.set_major_formatter(fmt_M)
ax4.grid(axis='y', linestyle='--', alpha=0.35, zorder=0)
ax4.spines[['top','right']].set_visible(False)
ax4.legend(handles=[Patch(color=GREEN, label='Real > Pronóstico'),
                    Patch(color=RED,   label='Real < Pronóstico')],
           fontsize=8, framealpha=0.7)

# ── E: Pronóstico 2026 ────────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[3, :])
ax5.set_facecolor(CARD)
ctx2 = ts[-12:]
ax5.plot(ctx2.index, ctx2.values/1e6, color=GRAY, lw=1.5,
         alpha=0.7, label='Real 2025')
ax5.fill_between(fc2_ci.index,
                 fc2_ci.iloc[:,0]/1e6, fc2_ci.iloc[:,1]/1e6,
                 color=GREEN, alpha=0.15, label='IC 95%')
ax5.plot(fc2.index, fc2.values/1e6,
         color=GREEN, lw=2.3, linestyle='--',
         marker='o', markersize=5, label='Pronóstico 2026')
ax5.axvline(pd.Timestamp('2026-01-01'), color=GRAY,
            lw=1, linestyle=':', alpha=0.6)
ax5.set_title('Pronóstico Gastos Administrativos 2026 (modelo 2021–2025)',
              fontsize=12, fontweight='bold', color=BLUE, pad=8)
ax5.set_ylabel('Millones MXN', fontsize=10)
ax5.yaxis.set_major_formatter(fmt_M)
ax5.xaxis.set_major_locator(mdates.MonthLocator())
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax5.tick_params(axis='x', labelsize=8)
ax5.legend(fontsize=9, framealpha=0.8)
ax5.grid(axis='y', linestyle='--', alpha=0.35)
ax5.spines[['top','right']].set_visible(False)
for date, val in fc2.items():
    ax5.text(date, val/1e6+0.2, f'${val/1e6:.1f}M',
             ha='center', va='bottom', fontsize=7.5,
             color=GREEN, fontweight='bold')

plt.savefig('gastos_admin.png',
            dpi=150, bbox_inches='tight', facecolor=BG)
print("\n✅  Guardado.")

print("\n── Pronóstico Gastos Administrativos 2026 ──")
for d, v in fc2.items():
    print(f"  {d.strftime('%b %Y')}: ${v:>12,.0f}")
print(f"\n  Total anual estimado 2026: ${fc2.sum():,.0f}")

Columnas seleccionadas:
  · Gastos de Mantenimiento
  · Gastos Administrativos
  · Gastos Financieros
  · Gastos no fiscales
  · ISR  Ejercicio
  · ISR Facilidades Administrativas
Train: 2021-07-01 → 2024-12-01 (42 meses)
Test : 2025-01-01 → 2025-12-01 (12 meses)

Ajustando SARIMA(1,1,1)(1,0,1)[12] …
                                     SARIMAX Results                                      
Dep. Variable:             Gastos Administrativos   No. Observations:                   42
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 12)   Log Likelihood                -436.833
Date:                            Sat, 21 Mar 2026   AIC                            883.665
Time:                                    01:04:11   BIC                            890.145
Sample:                                07-01-2021   HQIC                           885.592
                                     - 12-01-2024                                         
Covariance Type:                              opg            

In [6]:
print(df.columns.tolist())

['Fecha', 'Costo de Venta (Refacciones)', 'Costo de llantas', 'Costo de Lubricantes', 'Gastos de OperaciÃ³n', 'Gastos de Mantenimiento', 'Gastos Administrativos', 'Gastos Financieros', 'Gastos no fiscales', 'ISR  Ejercicio', 'ISR Facilidades Administrativas', 'Costo de lo Vendido']
